# Notebook 31 — Latency validation: analysis

**Author:** Sunita Sahu · **Revision item:** A5 (iii) latency validation (decision D10) · CPU only

Reads the per-model results of notebooks 29 (18 transformers-native models) and 30 (Molmo-7B-D, DeepSeek-VL2-Small) and answers five questions:

1. Does the cost model agree with an op-level FLOP count on the **full-depth released** checkpoints?
2. Which quantity tracks measured **prefill latency** across models: the predicted FLOPs, the parameter count, or the visual-token count?
3. Does the predicted **vision share of FLOPs** match the measured vision share of prefill time?
4. Within each model, does latency scale with the predicted FLOPs from 448 × 448 to 1024 × 1024?
5. What sets **decode** cost per token?
6. Is there a **per-pass floor** below which latency carries no information about FLOPs (decision log N28)? This section also counts the operator calls of one decode step on the CPU (needs transformers 4.57.6, the release of notebook 29) and reads notebook 33's counts for Gemma 4.

**Inputs.** Laptop: unzip the `ThreadB_GPU_outputs_*.zip` archives and copy `outputs/results/*.json` into `code_active/outputs/measurements/gpu_latency/results/` (keep `logs/`, `tables/` and `environment/` next to it in `gpu_latency/`). Colab: nothing to copy; the notebook reads `MyDrive/ThreadB_GPU/outputs/results/`.

**Outputs** (`nb31_*`): run overview, full-depth FLOP validation, latency table, proxy correlations, component shares, input scaling, decode table, the per-pass floor (prefill in decode steps, operator calls per decode step, decode against calls), and a plain-text summary of the numbers.

All correlations are Spearman ρ with two-sided permutation p-values (`correlation_grid.spearman_permutation_test`: exact up to n = 10, 20,000 Monte Carlo orderings above).

---
## Settings and setup

In [1]:
INCLUDE_DRY_RUN_RESULTS = False   # True only to test this notebook on CPU dry-run files
RESULTS_DIR_OVERRIDE = None       # a folder of result JSON files, if not the default location
# Results known to be invalid, with the reason (printed, never silently dropped), e.g.
# {'model name': 'reason'}. Empty since InstructBLIP was re-run with the fixed processor on 2026-10-03
# (decision log N20; the first result is kept in gpu_latency/superseded/).
EXCLUDED_RESULTS = {}
# N28: count the operator calls of one decode step (CPU, about 10 minutes; needs the transformers release
# of notebook 29). False, or another release, re-uses the saved nb31_operator_calls.csv.
COUNT_OPERATOR_CALLS = True

In [2]:
import os, sys, json, zipfile
from pathlib import Path
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_FOLDER = Path('/content/drive/MyDrive/ThreadB_GPU')
    CODE_DIR = Path('/content/threadb_code')
    with zipfile.ZipFile(PROJECT_FOLDER / 'ThreadB_GPU_bundle.zip') as archive:
        archive.extractall(CODE_DIR)
    sys.path.insert(0, str(CODE_DIR))
    RESULTS_DIR = PROJECT_FOLDER / 'outputs' / 'results'
    OUTPUT_DIR = PROJECT_FOLDER / 'outputs' / 'tables'
    LOG_DIR = PROJECT_FOLDER / 'outputs' / 'logs'
else:
    NOTEBOOK_DIR = Path.cwd()
    CODE_ACTIVE = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
    sys.path.insert(0, str(CODE_ACTIVE))
    RESULTS_DIR = CODE_ACTIVE / 'outputs' / 'measurements' / 'gpu_latency' / 'results'
    OUTPUT_DIR = CODE_ACTIVE / 'outputs' / 'measurements'
    LOG_DIR = CODE_ACTIVE / 'outputs' / 'logs'
RESULTS_DIR = Path(os.environ.get('THREADB_GPU_RESULTS') or RESULTS_DIR_OVERRIDE or RESULTS_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from vlm_audit.logging_utils import ExperimentLogger, SessionTranscript
from vlm_audit.latency_analysis import (TOKEN_BUDGET_MODELS, component_shares, decode_table,
                                       decode_versus_operator_calls, diagnostics_markdown, encoder_time_versus_flops,
                                       exclude_results, executed_versus_intrinsic, findings_markdown,
                                       floor_markdown, floor_summary, flop_validation_table, gpu_names,
                                       input_scaling, latency_rank_changes, latency_table,
                                       library_release_by_model, llm_based, load_results,
                                       matched_operator_counts, prefill_floor_table, prefill_in_decode_tokens,
                                       proxy_correlations, run_overview)
if globals().get('transcript') is not None:
    transcript.close()
transcript = SessionTranscript(LOG_DIR, '31_latency_analysis')
logger = ExperimentLogger(LOG_DIR, '31_latency_analysis')
print('results from:', RESULTS_DIR)
print('tables to   :', OUTPUT_DIR)

Session transcript: <repository>\code_active\outputs\logs\20261004_130237_31_latency_analysis__session_transcript.txt
Crash traces:       <repository>\code_active\outputs\logs\20261004_130237_31_latency_analysis__crash_traces.txt


<home>\AppData\Local\Temp\claude\<session>\0edba254-4773-49bc-b009-31568158ee2c\scratchpad\venv312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


results from: <repository>\code_active\outputs\measurements\gpu_latency\results
tables to   : <repository>\code_active\outputs\measurements


---
## 1. Results present, their status and the GPU they ran on

In [3]:
results = load_results(RESULTS_DIR, include_dry_run=INCLUDE_DRY_RUN_RESULTS)
if not results:
    raise FileNotFoundError(f'No results in {RESULTS_DIR}. Run notebooks 29 and 30 first (or copy their outputs here).')
overview = run_overview(results)
overview.to_csv(OUTPUT_DIR / 'nb31_run_overview.csv', index=False)   # every result, excluded ones included
results = exclude_results(results, EXCLUDED_RESULTS)
GPUS = gpu_names(results)
logger.info(f'{len(results)} results; GPUs {GPUS}')
if len(GPUS) > 1:
    print(f'WARNING: results come from {len(GPUS)} GPU types {GPUS}; latencies are only comparable within one GPU. '
          'Re-run the minority models on the majority GPU before using the latency tables.')
print(f'{len(results)} models; GPUs: {GPUS}')
overview

23 models; GPUs: ['NVIDIA A100-SXM4-80GB']


,model,status,mode,gpu,transformers,torch,attention,inputs_complete,stage_errors,run_error,notes
0,BLIP-2-OPT-2.7B,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,
1,BLIP-large,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,eager,"448x448, 1024x1024",,,
2,Chameleon-7B,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,forward ignores logits_to_keep: LM-head logits...
3,CLIP-ViT-L/14,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,
4,DeepSeek-VL2-Small,complete,released,NVIDIA A100-SXM4-80GB,4.46.3,2.11.0+cu130,eager,"448x448, 1024x1024",,,released code computes latent attention in the...
5,Gemma-4-12B,complete,released,NVIDIA A100-SXM4-80GB,5.18.0,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,processor class: Gemma4UnifiedProcessor
6,Gemma-4-26B-A4B,complete,released,NVIDIA A100-SXM4-80GB,5.18.0,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,processor class: Gemma4Processor | experts: ti...
7,IDEFICS-9B,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,
8,Idefics2-8B,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,
9,InstructBLIP-Vicuna-7B,complete,released,NVIDIA A100-SXM4-80GB,4.57.6,2.11.0+cu130,sdpa,"448x448, 1024x1024",,,processor class: InstructBlipProcessor


---
## 2. Cost model against the full-depth op-level FLOP count

Notebook 27 validated the cost model on reduced-depth copies of 18 architectures (largest difference 0.008%). Here the same comparison runs on the released, full-depth checkpoints, on the executed input (padding tiles and padded sub-images included), and for the first time on Molmo-7B-D and DeepSeek-VL2-Small.

In [4]:
validation = flop_validation_table(results)
validation.to_csv(OUTPUT_DIR / 'nb31_flop_validation_full_depth.csv', index=False)
components_only = validation[validation.component != 'total']
if not components_only.empty:
    worst = components_only.loc[components_only.relative_difference_pct.abs().idxmax()]
    message = (f'{len(components_only)} component comparisons; largest |difference| '
               f'{abs(worst.relative_difference_pct):.4f}% ({worst.model}, {worst.input}, {worst.component})')
    print(message); logger.metric('flop_validation_largest_abs_difference_pct', abs(worst.relative_difference_pct))
validation.pivot_table(index=['model', 'input'], columns='component', values='relative_difference_pct').round(4)

168 component comparisons; largest |difference| 0.0077% (Chameleon-7B, 448x448, language)


component                         fusion  head  language   total  vision
model                  input                                            
BLIP-2-OPT-2.7B        1024x1024     0.0  -0.0    0.0000  0.0000  0.0000
                       448x448       0.0  -0.0    0.0000  0.0000  0.0000
BLIP-large             1024x1024     NaN   0.0    0.0000  0.0000  0.0000
                       448x448       NaN   0.0    0.0000  0.0000  0.0000
CLIP-ViT-L/14          1024x1024     0.0   NaN    0.0000 -0.0000  0.0000
                       448x448       0.0   NaN    0.0000 -0.0000  0.0000
Chameleon-7B           1024x1024     NaN   0.0    0.0077  0.0072  0.0000
                       448x448       NaN   0.0    0.0077  0.0072  0.0000
DeepSeek-VL2-Small     1024x1024     0.0  -0.0    0.0000  0.0000  0.0000
                       448x448       0.0  -0.0    0.0000  0.0000  0.0000
Gemma-4-12B            1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Gemma-4-26B-A4B        1024x1024     0.0   0.0   -0.0000 -0.0000 -0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000 -0.0000
IDEFICS-9B             1024x1024     0.0   0.0    0.0000 -0.0000  0.0000
                       448x448       0.0   0.0    0.0000 -0.0000  0.0000
Idefics2-8B            1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
InstructBLIP-Vicuna-7B 1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
InternVL3-8B           1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Janus-Pro-7B           1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
LLaVA-1.5-7B           1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
LLaVA-NeXT-7B          1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Llama-3.2-11B-Vision   1024x1024     0.0   0.0   -0.0000 -0.0004 -0.0004
                       448x448       0.0   0.0   -0.0000 -0.0004 -0.0004
Molmo-7B-D             1024x1024     0.0   0.0    0.0000  0.0000  0.0000
                       448x448       0.0   0.0    0.0000  0.0000  0.0000
Pixtral-12B            1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Qwen2-VL-7B            1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Qwen2.5-VL-7B          1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
Qwen3-VL-8B            1024x1024     0.0   0.0   -0.0000 -0.0000  0.0000
                       448x448       0.0   0.0   -0.0000 -0.0000  0.0000
SigLIP-L/16            1024x1024     NaN   NaN    0.0000 -0.0000  0.0000
                       448x448       NaN   NaN    0.0000 -0.0000  0.0000
SigLIP2-So400m/14      1024x1024     NaN   NaN    0.0000 -0.0000  0.0000
                       448x448       NaN   NaN    0.0000 -0.0000  0.0000
ViLT-B/32              1024x1024     NaN   0.0    0.0000  0.0000  0.0000
                       448x448       NaN   0.0    0.0000  0.0000  0.0000

---
## 3. Latency table (1024 × 1024 is the primary input)

In [5]:
table = latency_table(results)
table.to_csv(OUTPUT_DIR / 'nb31_latency_table.csv', index=False)
columns = ['model', 'input', 'language_visual_tokens', 'predicted_total_tflops', 'prefill_ms_median', 'prefill_ms_q1',
           'prefill_ms_q3', 'achieved_tflops_per_second', 'vision_time_share_pct', 'predicted_vision_share_pct',
           'decode_ms_per_token', 'weights_resident_gb', 'prefill_activation_peak_gb', 'elementwise_share_pct']
table[columns].sort_values(['input', 'prefill_ms_median']).round(2)

,model,input,language_visual_tokens,predicted_total_tflops,prefill_ms_median,prefill_ms_q1,prefill_ms_q3,achieved_tflops_per_second,vision_time_share_pct,predicted_vision_share_pct,decode_ms_per_token,weights_resident_gb,prefill_activation_peak_gb,elementwise_share_pct
45,ViLT-B/32,1024x1024,145,0.03,12.38,12.36,12.41,2.23,24.59,2.47,NaN,0.25,0.02,0.35
7,CLIP-ViT-L/14,1024x1024,0,0.16,20.53,20.44,20.55,7.96,62.59,99.17,NaN,0.86,0.01,0.17
41,SigLIP-L/16,1024x1024,0,0.42,24.55,24.50,24.65,17.23,54.83,90.76,NaN,1.32,0.02,0.19
43,SigLIP2-So400m/14,1024x1024,0,0.72,28.47,27.84,29.08,25.42,53.10,92.66,NaN,2.28,0.02,0.16
3,BLIP-large,1024x1024,0,0.41,33.06,32.80,33.62,12.28,41.60,94.11,16.59,0.90,0.03,0.37
1,BLIP-2-OPT-2.7B,1024x1024,32,0.75,53.53,52.49,53.80,13.93,30.27,69.83,21.17,7.50,0.02,0.11
23,Janus-Pro-7B,1024x1024,578,8.15,63.21,63.11,63.45,128.93,23.67,4.68,34.95,14.83,0.37,0.04
27,LLaVA-1.5-7B,1024x1024,576,8.16,63.22,62.97,63.30,129.11,20.62,4.68,35.80,14.14,0.37,0.03
19,InstructBLIP-Vicuna-7B,1024x1024,32,1.06,74.24,74.06,74.62,14.34,22.90,48.92,36.59,15.83,0.03,0.09
15,IDEFICS-9B,1024x1024,0,0.55,78.01,77.08,78.39,7.04,20.63,60.89,54.92,17.91,0.01,0.12


---
## 4. Which quantity tracks prefill latency?

The submitted text (Secs. 5.3 and 6.1) says parameter counts are a poor proxy for forward cost and that the visual tokens entering the language model, times its size, set the cost. This is the test against measured time.

In [6]:
proxies = proxy_correlations(table)
proxies.to_csv(OUTPUT_DIR / 'nb31_latency_proxies.csv', index=False)
for _, row in proxies.iterrows():
    logger.metric(f"rho_{row['input']}_{row['proxy']}", round(row['spearman_rho'], 4))
proxies.round(4)

,input,response,proxy,description,sample_size,spearman_rho,p_value,exact,log_log_slope,log_log_intercept,log_log_r_squared
0,1024x1024,prefill_ms_median,predicted_total_gflops,"cost model, executed input",23,0.8666,0.0001,False,0.4590,0.3510,0.8334
1,1024x1024,prefill_ms_median,predicted_intrinsic_total_gflops,"cost model, revised Table 3 convention",23,0.8409,0.0001,False,0.4731,0.2974,0.8180
2,1024x1024,prefill_ms_median,parameter_count,total parameters,23,0.8231,0.0000,False,0.6772,-4.5026,0.7101
3,1024x1024,prefill_ms_median,language_visual_tokens,visual tokens in the language model,23,0.6581,0.0011,False,NaN,NaN,NaN
4,448x448,prefill_ms_median,predicted_total_gflops,"cost model, executed input",23,0.8093,0.0000,False,0.4013,0.4975,0.7065
5,448x448,prefill_ms_median,predicted_intrinsic_total_gflops,"cost model, revised Table 3 convention",23,0.7095,0.0003,False,0.4056,0.4858,0.6476
6,448x448,prefill_ms_median,parameter_count,total parameters,23,0.8083,0.0000,False,0.5440,-3.3726,0.7497
7,448x448,prefill_ms_median,language_visual_tokens,visual tokens in the language model,23,0.5782,0.0042,False,NaN,NaN,NaN


---
## 5. Vision share of prefill time against the predicted vision share of FLOPs

In [7]:
shares = component_shares(table)
shares.to_csv(OUTPUT_DIR / 'nb31_component_shares.csv', index=False)
shares.round(3)

,input,sample_size,spearman_rho,p_value,mean_difference_pp,mean_absolute_difference_pp,largest_absolute_difference_pp,largest_difference_model
0,1024x1024,23,0.736,0.000,-7.652,22.428,52.512,BLIP-large
1,448x448,23,0.677,0.001,-7.409,23.357,53.620,BLIP-large


---
## 6. Within-model scaling from 448 × 448 to 1024 × 1024

In [8]:
scaling = input_scaling(table)
scaling['per_model'].to_csv(OUTPUT_DIR / 'nb31_input_scaling.csv', index=False)
print({key: value for key, value in scaling.items() if key != 'per_model'})
scaling['per_model'][['model', 'predicted_flop_ratio', 'latency_ratio', 'decode_ratio']].round(3)

{'models_whose_flops_change': 8, 'spearman_rho': 0.3095238095238094, 'p_value': 0.4618055555555556}


,model,predicted_flop_ratio,latency_ratio,decode_ratio
0,BLIP-2-OPT-2.7B,1.000,0.988,1.006
1,BLIP-large,1.000,0.980,1.000
2,Chameleon-7B,1.000,1.004,1.001
3,CLIP-ViT-L/14,1.000,0.965,NaN
4,DeepSeek-VL2-Small,2.173,1.177,1.002
5,Gemma-4-12B,1.000,0.997,1.005
6,Gemma-4-26B-A4B,1.000,0.981,1.001
7,IDEFICS-9B,1.000,1.032,1.006
8,Idefics2-8B,4.094,5.813,1.004
9,InstructBLIP-Vicuna-7B,1.000,1.005,0.999


---
## 7. Decode cost per token

In [9]:
decode = decode_table(table)
decode['per_model'].to_csv(OUTPUT_DIR / 'nb31_decode.csv', index=False)
print({key: value for key, value in decode.items() if key != 'per_model'})
decode['per_model'].sort_values('decode_ms_per_token').round(3)

{'rho_with_weights_resident_gb': 0.7894736842105262, 'p_with_weights_resident_gb': 0.000199990000499975, 'n_with_weights_resident_gb': 19, 'rho_with_active_parameter_count': 0.375438596491228, 'p_with_active_parameter_count': 0.11874406279686016, 'n_with_active_parameter_count': 19, 'rho_with_language_visual_tokens': 0.06514124889314374, 'p_with_language_visual_tokens': 0.783910804459777, 'n_with_language_visual_tokens': 19}


,model,decode_ms_per_token,weights_resident_gb,active_parameter_count,parameter_count,language_visual_tokens
1,BLIP-large,16.593,0.903,4.462905e+08,446290492,0
0,BLIP-2-OPT-2.7B,21.170,7.500,3.744762e+09,3744761856,32
9,InternVL3-8B,34.298,15.899,7.944374e+09,7944373760,2562
10,Janus-Pro-7B,34.952,14.826,7.407773e+09,7407773323,578
12,LLaVA-1.5-7B,35.799,14.136,7.063427e+09,7063427072,576
8,InstructBLIP-Vicuna-7B,36.591,15.828,7.913727e+09,7913726720,32
13,LLaVA-NeXT-7B,37.019,15.143,7.566748e+09,7566747648,2928
16,Qwen2-VL-7B,37.358,16.593,8.291376e+09,8291375616,1371
17,Qwen2.5-VL-7B,37.730,16.594,8.292167e+09,8292166656,1371
7,Idefics2-8B,37.983,16.815,8.402768e+09,8402768112,326


---
## 8. Diagnostics

Added after the first GPU run. (a) The proxy test on the LLM-based models only, so that a proxy cannot win merely by separating sub-billion encoders from 7B language models. (b) Executed versus intrinsic FLOPs: the padding reference implementations run. (c) Whether the cheaper model of a pair changes with the input size. (d) The vision encoder's share of prefill time against its share of FLOPs, and the throughput each component reaches. (e) Prefill cost in units of decode steps.

In [10]:
llm_proxies = proxy_correlations(llm_based(table))
llm_proxies.to_csv(OUTPUT_DIR / 'nb31_latency_proxies_llm_based.csv', index=False)
llm_proxies.round(4)

,input,response,proxy,description,sample_size,spearman_rho,p_value,exact,log_log_slope,log_log_intercept,log_log_r_squared
0,1024x1024,prefill_ms_median,predicted_total_gflops,"cost model, executed input",18,0.7399,0.0007,False,0.3989,0.6165,0.6085
1,1024x1024,prefill_ms_median,predicted_intrinsic_total_gflops,"cost model, revised Table 3 convention",18,0.6739,0.0027,False,0.4173,0.5420,0.5546
2,1024x1024,prefill_ms_median,parameter_count,total parameters,18,0.6367,0.0052,False,1.0183,-7.8991,0.2866
3,1024x1024,prefill_ms_median,language_visual_tokens,visual tokens in the language model,18,0.4780,0.0450,False,NaN,NaN,NaN
4,448x448,prefill_ms_median,predicted_total_gflops,"cost model, executed input",18,0.6244,0.0060,False,0.2921,0.9335,0.3194
5,448x448,prefill_ms_median,predicted_intrinsic_total_gflops,"cost model, revised Table 3 convention",18,0.3994,0.1037,False,0.2521,1.0885,0.1857
6,448x448,prefill_ms_median,parameter_count,total parameters,18,0.6058,0.0087,False,1.0302,-8.2197,0.4776
7,448x448,prefill_ms_median,language_visual_tokens,visual tokens in the language model,18,0.3476,0.1593,False,NaN,NaN,NaN


In [11]:
executed = executed_versus_intrinsic(table)
executed.to_csv(OUTPUT_DIR / 'nb31_executed_versus_intrinsic.csv', index=False)
executed.round(2)

,model,input,predicted_total_gflops,predicted_intrinsic_total_gflops,encoder_tokens,prefill_ms_median,executed_over_intrinsic
14,IDEFICS-9B,448x448,549.54,876.28,257,75.58,0.63
15,IDEFICS-9B,1024x1024,549.54,876.28,257,78.01,0.63
18,InstructBLIP-Vicuna-7B,448x448,1064.47,1370.56,257,73.84,0.78
19,InstructBLIP-Vicuna-7B,1024x1024,1064.47,1370.56,257,74.24,0.78
45,ViLT-B/32,1024x1024,27.55,31.91,0,12.38,0.86
44,ViLT-B/32,448x448,27.55,31.91,0,12.08,0.86
0,BLIP-2-OPT-2.7B,448x448,745.75,857.32,257,54.21,0.87
1,BLIP-2-OPT-2.7B,1024x1024,745.75,857.32,257,53.53,0.87
38,Qwen3-VL-8B,448x448,3646.36,4014.19,784,89.14,0.91
11,Gemma-4-12B,1024x1024,5844.25,6422.28,0,111.48,0.91


In [12]:
rank_changes = latency_rank_changes(table)
rank_changes['per_model'].to_csv(OUTPUT_DIR / 'nb31_latency_by_input.csv', index=False)
rank_changes['flipped_pairs'].to_csv(OUTPUT_DIR / 'nb31_latency_order_flips.csv', index=False)
print(f"Kendall tau {rank_changes['kendall_tau']:.2f}; {len(rank_changes['flipped_pairs'])} of {rank_changes['pairs']} pairs change order")
rank_changes['flipped_pairs'].round(1)

Kendall tau 0.71; 32 of 253 pairs change order


,first,second,first_ms_448x448,second_ms_448x448,first_ms_1024x1024,second_ms_1024x1024
0,BLIP-2-OPT-2.7B,InternVL3-8B,54.2,49.4,53.5,233.8
1,Chameleon-7B,Idefics2-8B,114.2,96.6,114.7,561.6
2,Chameleon-7B,InternVL3-8B,114.2,49.4,114.7,233.8
3,Chameleon-7B,Qwen2-VL-7B,114.2,72.6,114.7,213.6
4,Chameleon-7B,Qwen2.5-VL-7B,114.2,98.9,114.7,309.8
5,Chameleon-7B,Qwen3-VL-8B,114.2,89.1,114.7,161.4
6,DeepSeek-VL2-Small,Idefics2-8B,354.6,96.6,417.5,561.6
7,DeepSeek-VL2-Small,Pixtral-12B,354.6,134.0,417.5,575.2
8,Gemma-4-12B,Idefics2-8B,111.8,96.6,111.5,561.6
9,Gemma-4-12B,InternVL3-8B,111.8,49.4,111.5,233.8


In [13]:
encoder = encoder_time_versus_flops(table)
encoder.to_csv(OUTPUT_DIR / 'nb31_encoder_time_versus_flops.csv', index=False)
encoder.round(2)

,model,predicted_vision_share_pct,vision_time_share_pct,time_share_over_flop_share,encoder_tflops_per_second,language_tflops_per_second,achieved_tflops_per_second
0,IDEFICS-9B,60.89,20.63,0.34,20.79,3.21,7.04
1,DeepSeek-VL2-Small,32.41,11.66,0.36,136.90,36.00,49.25
2,Gemma-4-26B-A4B,67.95,24.87,0.37,53.94,8.82,19.75
3,BLIP-2-OPT-2.7B,69.83,30.27,0.43,32.14,9.49,13.93
4,InstructBLIP-Vicuna-7B,48.92,22.90,0.47,30.63,13.84,14.34
5,Llama-3.2-11B-Vision,92.87,81.17,0.87,82.40,20.59,72.02
6,Idefics2-8B,77.62,86.93,1.12,71.92,116.01,80.55
7,InternVL3-8B,16.62,20.39,1.23,151.74,196.65,186.19
8,Qwen2-VL-7B,36.74,50.21,1.37,110.21,190.45,150.62
9,Qwen3-VL-8B,26.34,38.01,1.44,89.16,156.06,128.66


In [14]:
prefill_decode = prefill_in_decode_tokens(table)
prefill_decode.to_csv(OUTPUT_DIR / 'nb31_prefill_in_decode_tokens.csv', index=False)
prefill_decode.round(1)

,model,prefill_ms_median,decode_ms_per_token,prefill_in_decode_tokens,weights_resident_gb
0,Gemma-4-12B,111.5,108.2,1.0,23.9
1,IDEFICS-9B,78.0,54.9,1.4,17.9
2,LLaVA-1.5-7B,63.2,35.8,1.8,14.1
3,Janus-Pro-7B,63.2,35.0,1.8,14.8
4,BLIP-large,33.1,16.6,2.0,0.9
5,InstructBLIP-Vicuna-7B,74.2,36.6,2.0,15.8
6,Chameleon-7B,114.7,49.6,2.3,14.1
7,BLIP-2-OPT-2.7B,53.5,21.2,2.5,7.5
8,Gemma-4-26B-A4B,264.0,101.5,2.6,51.6
9,Qwen3-VL-8B,161.4,55.1,2.9,17.5


---
## 9. The per-pass floor (decision log N28)

At batch size one a reference implementation issues one GPU kernel per operator call, and the host issues them one at a time. A pass therefore cannot be faster than its operator calls allow, whatever its FLOPs. Three parts:

(a) **Prefill against the floor.** A decode step is one pass over one token, so it measures the floor. The language model's prefill time is expressed in decode steps against the number of tokens it sees. Mixture-of-experts models are shown apart: a prefill over many tokens reaches most experts, a decode step only a few.

(b) **Decode against operator calls.** `operator_dispatch.count_decode_step` counts the kernel-launching calls of one decode step on the reduced-depth real architecture, with the library's default attention and expert kernels, and scales them to full depth by layer type. Each model is counted under the transformers release of its GPU measurement: 4.57.6 here, 5.18.0 for Gemma 4 in notebook 33. Qwen3-VL-8B is counted under both. Molmo-7B-D and DeepSeek-VL2-Small (remote code, 4.46.3) are not counted.

(c) The prefill correlations of section 4 without the two token-budget models, which sit on the floor at every image size.

In [15]:
floor = prefill_floor_table(table)
floor.to_csv(OUTPUT_DIR / 'nb31_prefill_floor.csv', index=False)
floor_overview = floor_summary(floor)
print({key: value for key, value in floor_overview.items() if key != 'mixture_of_experts_rows'})
floor.round(2)

{'dense_measurements': 34, 'on_floor': 18, 'tolerance_decode_steps': 1.2, 'on_floor_ratio_min': 0.9771924256537289, 'on_floor_ratio_max': 1.153478964647701, 'on_floor_max_tokens': 334, 'above_floor_min_tokens': 585, 'separated_by_token_count': True}


,model,input,language_tokens,language_ms_median,decode_ms_per_token,language_prefill_in_decode_steps,on_floor,mixture_of_experts
0,BLIP-large,1024x1024,8,19.08,16.59,1.15,True,False
1,BLIP-large,448x448,8,19.13,16.59,1.15,True,False
2,Llama-3.2-11B-Vision,1024x1024,9,49.68,45.89,1.08,True,False
3,Llama-3.2-11B-Vision,448x448,9,49.32,45.89,1.07,True,False
4,IDEFICS-9B,1024x1024,11,57.00,54.92,1.04,True,False
5,IDEFICS-9B,448x448,11,56.83,54.61,1.04,True,False
6,BLIP-2-OPT-2.7B,1024x1024,40,21.26,21.17,1.00,True,False
7,BLIP-2-OPT-2.7B,448x448,40,22.03,21.04,1.05,True,False
8,InstructBLIP-Vicuna-7B,1024x1024,40,37.50,36.59,1.02,True,False
9,InstructBLIP-Vicuna-7B,448x448,40,37.71,36.64,1.03,True,False


In [16]:
import transformers
from transformers.utils import logging as transformers_logging
from vlm_audit.operator_dispatch import BRIDGE_MODEL, TRANSFORMERS4_DECODE_MODELS, decode_step_table

native_release = library_release_by_model(results).get('LLaVA-1.5-7B')
counts_path = OUTPUT_DIR / 'nb31_operator_calls.csv'
if COUNT_OPERATOR_CALLS and transformers.__version__ == native_release:
    transformers_logging.set_verbosity_error()
    operator_counts = decode_step_table(TRANSFORMERS4_DECODE_MODELS, logger=logger)
    operator_counts.to_csv(counts_path, index=False)
elif counts_path.exists():
    operator_counts = pd.read_csv(counts_path)
    print(f'transformers {transformers.__version__} (GPU run: {native_release}): re-using {counts_path.name}')
else:
    operator_counts = pd.DataFrame(columns=['model', 'status', 'transformers_version', 'calls_per_decode_step'])
    print(f'No operator counts: run this notebook with transformers {native_release} to count them.')
transformers5_counts_path = OUTPUT_DIR / 'nb33_operator_calls.csv'
if transformers5_counts_path.exists():
    operator_counts = pd.concat([operator_counts, pd.read_csv(transformers5_counts_path)], ignore_index=True)
else:
    print('nb33_operator_calls.csv not found: Gemma 4 is not counted (run notebook 33 first).')
matched_counts = matched_operator_counts(operator_counts, results)
bridge = operator_counts[(operator_counts['model'] == BRIDGE_MODEL) & (operator_counts['status'] == 'complete')]
dispatch = decode_versus_operator_calls(table, matched_counts)
dispatch['per_model'].to_csv(OUTPUT_DIR / 'nb31_decode_versus_operator_calls.csv', index=False)
print({key: value for key, value in dispatch.items() if key != 'per_model'})
dispatch['per_model'].round(2)

BLIP-large: 370 calls per decode step (12 blocks; {"BlipTextLayer": 26})


BLIP-2-OPT-2.7B: 523 calls per decode step (32 blocks; {"OPTDecoderLayer": 15})


InstructBLIP-Vicuna-7B: 1337 calls per decode step (32 blocks; {"LlamaDecoderLayer": 40})


LLaVA-1.5-7B: 1340 calls per decode step (32 blocks; {"LlamaDecoderLayer": 40})


LLaVA-NeXT-7B: 1340 calls per decode step (32 blocks; {"MistralDecoderLayer": 40})


IDEFICS-9B: 1906 calls per decode step (40 blocks; {"IdeficsDecoderLayer": 45, "IdeficsGatedCrossAttentionLayer": 50})


... (7 outputs left out)

Chameleon-7B: 1779 calls per decode step (32 blocks; {"ChameleonDecoderLayer": 54})


Janus-Pro-7B: 1260 calls per decode step (30 blocks; {"LlamaDecoderLayer": 40})


{'sample_size': 17, 'not_counted': ['DeepSeek-VL2-Small', 'Molmo-7B-D'], 'spearman_with_calls_per_decode_step': 0.9231758405900331, 'p_with_calls_per_decode_step': 4.999750012499375e-05, 'pearson_with_calls_per_decode_step': 0.9952974274722215, 'spearman_with_weights_resident_gb': 0.7892156862745098, 'p_with_weights_resident_gb': 0.00034998250087495624, 'pearson_with_weights_resident_gb': 0.7957150324155727, 'spearman_with_active_parameter_count': 0.5686274509803922, 'p_with_active_parameter_count': 0.01814909254537273, 'pearson_with_active_parameter_count': 0.36187701432442976, 'fit_microseconds_per_call': 21.247684335137013, 'fit_intercept_ms': 9.96300048820612}


,model,decode_ms_per_token,weights_resident_gb,active_parameter_count,calls_per_decode_step,transformers_version,host_synchronisations_reduced,microseconds_per_call
0,BLIP-large,16.59,0.90,4.462905e+08,370,4.57.6,2,44.85
1,BLIP-2-OPT-2.7B,21.17,7.50,3.744762e+09,523,4.57.6,2,40.48
2,InternVL3-8B,34.30,15.90,7.944374e+09,1184,4.57.6,6,28.97
3,Qwen2.5-VL-7B,37.73,16.59,8.292167e+09,1242,4.57.6,5,30.38
4,Qwen2-VL-7B,37.36,16.59,8.291376e+09,1242,4.57.6,5,30.08
5,Janus-Pro-7B,34.95,14.83,7.407773e+09,1260,4.57.6,4,27.74
6,InstructBLIP-Vicuna-7B,36.59,15.83,7.913727e+09,1337,4.57.6,2,27.37
7,Idefics2-8B,37.98,16.82,8.402768e+09,1340,4.57.6,4,28.35
8,LLaVA-NeXT-7B,37.02,15.14,7.566748e+09,1340,4.57.6,4,27.63
9,LLaVA-1.5-7B,35.80,14.14,7.063427e+09,1340,4.57.6,4,26.72


In [17]:
budget_free_proxies = proxy_correlations(table[~table['model'].isin(TOKEN_BUDGET_MODELS)])
budget_free_proxies = budget_free_proxies[budget_free_proxies['proxy'].isin(['predicted_total_gflops',
                                                                            'parameter_count'])]
budget_free_proxies.round(4)

,input,response,proxy,description,sample_size,spearman_rho,p_value,exact,log_log_slope,log_log_intercept,log_log_r_squared
0,1024x1024,prefill_ms_median,predicted_total_gflops,"cost model, executed input",21,0.9130,0.0,False,0.4586,0.3366,0.8569
2,1024x1024,prefill_ms_median,parameter_count,total parameters,21,0.8714,0.0,False,0.7098,-4.7969,0.7233
4,448x448,prefill_ms_median,predicted_total_gflops,"cost model, executed input",21,0.8571,0.0,False,0.3911,0.5096,0.7461
6,448x448,prefill_ms_median,parameter_count,total parameters,21,0.7974,0.0,False,0.5324,-3.2640,0.7272


---
## 10. Summary of the numbers and closing the logs

In [18]:
summary_text = findings_markdown(results, validation, table, proxies, shares, scaling, decode)
summary_text += diagnostics_markdown(llm_proxies, executed, rank_changes, encoder, prefill_decode, EXCLUDED_RESULTS)
summary_text += floor_markdown(floor_overview, dispatch, budget_free_proxies, bridge)
(OUTPUT_DIR / 'nb31_latency_findings.md').write_text(summary_text, encoding='utf-8')
print(summary_text)
logger.close()
transcript.close()
transcript = None

# Latency validation — summary of notebook 31

- Models with results: 23; GPUs: NVIDIA A100-SXM4-80GB.
- Cost model vs full-depth op-level FLOPs: 168 component comparisons; largest |difference| 0.008% (Chameleon-7B, 448x448, language).
- Element-wise work at full depth: 0.02–0.37% of FLOPs (median 0.09%).
- 1024x1024: prefill latency vs cost model, executed input: Spearman ρ = +0.87 (n = 23, p < 0.0001); log-log slope 0.46, R² 0.83.
- 1024x1024: prefill latency vs cost model, revised Table 3 convention: Spearman ρ = +0.84 (n = 23, p < 0.0001); log-log slope 0.47, R² 0.82.
- 1024x1024: prefill latency vs total parameters: Spearman ρ = +0.82 (n = 23, p < 0.0001); log-log slope 0.68, R² 0.71.
- 1024x1024: prefill latency vs visual tokens in the language model: Spearman ρ = +0.66 (n = 23, p = 0.0011).
- 448x448: prefill latency vs cost model, executed input: Spearman ρ = +0.81 (n = 23, p < 0.0001); log-log slope 0.40, R² 0.71.
- 448x448: prefill latency vs cost model, revised Table 3 conve